# LLAMA INDEX

In [1]:
from llama_index.llms.ollama import Ollama

from llama_index.core.workflow import Context

In [2]:
import asyncio
from llama_index.core.agent.workflow import FunctionAgent



# Define a simple calculator tool
def multiply(a: float, b: float) -> float:
    """Useful for multiplying two numbers."""
    return a * b


# Create an agent workflow with our calculator tool
agent = FunctionAgent(
    tools=[multiply],
    llm=Ollama(
        model="llama3.2",
        request_timeout=360.0,
        # Manually set the context window to limit memory usage
        context_window=8000,
    ),
    system_prompt="You are a helpful assistant that can multiply two numbers.",
)


In [3]:
ctx = Context(agent)

In [4]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.embeddings.ollama import OllamaEmbedding
import os

/Users/eddieogola/dev/projects/ain/server/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Settings control global defaults
Settings.embed_model = OllamaEmbedding(
    model_name="embeddinggemma",
    request_timeout=360.0,
)
Settings.llm = Ollama(
    model="llama3.2",
    request_timeout=360.0,
    # Manually set the context window to limit memory usage
    context_window=8000,
)

In [10]:
# Create a RAG tool using LlamaIndex
documents = SimpleDirectoryReader("data").load_data()
print(documents[0].id_)

f310c2c0-d8cc-47fd-8e8f-adbb35c66e65


In [ ]:
# # Indexing
# index = VectorStoreIndex.from_documents(
#     documents,
#     # we can optionally override the embed_model here
#     embed_model=Settings.embed_model,
# )
# # Save the index
# index.storage_context.persist("storage")

In [6]:

# Later, load the index
from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(persist_dir="storage")
index = load_index_from_storage(
    storage_context,
    # we can optionally override the embed_model here
    # it's important to use the same embed_model as the one used to build the index
    embed_model=Settings.embed_model,
)

Loading llama_index.core.storage.kvstore.simple_kvstore from storage/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from storage/index_store.json.


In [14]:
query_engine = index.as_query_engine(streaming=True, similarity_top_k=3)


In [15]:
response = query_engine.query("What is attention in transformers")
response.print_response_stream()

Attention mechanisms allow models to focus on specific parts of input data when generating output. This process helps to identify long-distance dependencies within a sentence, like the connection between the verb "making" and the phrase "more difficult".

# LANGCHAIN

In [1]:
import sys
sys.path.append('../src')

In [2]:
from config import get_config

/Users/eddieogola/dev/projects/ain/server/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
config = get_config()

In [4]:
embeddings = config.embed_model

In [5]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="ain_docs_v1",
    embedding_function=embeddings,
    persist_directory="./data/storage",  # Where to save data locally, remove if not necessary
)

In [6]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "./data/uploads/1706.03762v7.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

15


In [7]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need


{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/uploads/1706.03762v7.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

len(all_splits)

52

In [9]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[0.021153967827558517, 0.005655569024384022, 0.008207947015762329, -0.011677960865199566, -0.004303682595491409, -0.02860841527581215, -0.026981765404343605, -0.019237060099840164, -0.007933983579277992, -0.009035197086632252]


In [10]:
ids = vector_store.add_documents(documents=all_splits)

In [11]:
results = vector_store.similarity_search(
    "What is attention in transformers?",k=3
)

print(results[0])

page_content='Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
Figure 5: Many of the attention heads exhibit behaviour that seems related to the structure of the
sentence. We give two such examples above, from two different heads from the encoder self-attention
at layer 5 of 6. The heads clearly learned to perform different tasks.
15' metadata={'page_label': '15', 'producer': 'pdfTeX-1.40.25', 'page': 14, 'title': '', 'moddate': '2024-04-10T21:11:43+00:00', 'start_index': 0, 'creator': 'LaTeX with 

In [12]:
results = await vector_store.asimilarity_search("When was Nike incorporated?", k=3)

print(results[0])

page_content='and interpretable tree annotation. In Proceedings of the 21st International Conference on
Computational Linguistics and 44th Annual Meeting of the ACL, pages 433–440. ACL, July
2006.
[30] Ofir Press and Lior Wolf. Using the output embedding to improve language models. arXiv
preprint arXiv:1608.05859, 2016.
[31] Rico Sennrich, Barry Haddow, and Alexandra Birch. Neural machine translation of rare words
with subword units. arXiv preprint arXiv:1508.07909, 2015.
[32] Noam Shazeer, Azalia Mirhoseini, Krzysztof Maziarz, Andy Davis, Quoc Le, Geoffrey Hinton,
and Jeff Dean. Outrageously large neural networks: The sparsely-gated mixture-of-experts
layer. arXiv preprint arXiv:1701.06538, 2017.
[33] Nitish Srivastava, Geoffrey E Hinton, Alex Krizhevsky, Ilya Sutskever, and Ruslan Salakhutdi-
nov. Dropout: a simple way to prevent neural networks from overfitting. Journal of Machine
Learning Research, 15(1):1929–1958, 2014.' metadata={'author': '', 'creator': 'LaTeX with hyperref', 'm

In [13]:
embedding = embeddings.embed_query("How were Nike's margins impacted in 2023?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

page_content='Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network. We employ a residual connection [11] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel = 512.' metadata={'moddate': '2024-04-10T21:11:43+00:00', 'keywords'

In [14]:
len(results)

4

In [15]:
lk = ["today", "is here"]
"\n\n".join(lk)

'today\n\nis here'

In [16]:
result

NameError: name 'result' is not defined

In [ ]:
retrieved_context = list(map(lambda result: result.page_content, results))

In [ ]:
print("\n\n".join(retrieved_context))

In [36]:
from agents.rag import agent 

In [37]:
query = (
    "Who are the authors of the article?\n\n")

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Who are the authors of the article?


================================== Ai Message ==================================
Tool Calls:
  retrieve_context (J1xWAgoM0edbukZbthm7xGLNkqaGPSLc)
 Call ID: J1xWAgoM0edbukZbthm7xGLNkqaGPSLc
  Args:
    query: Who are the authors of the article?
================================= Tool Message =================================
Name: retrieve_context

Source: {'subject': '', 'page': 6, 'creationdate': '2022-09-29T00:18:07+00:00', 'moddate': '2022-09-29T00:18:07+00:00', 'total_pages': 10, 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'source': '/app/rag/data/uploads/2209.07663v2.pdf', 'creator': 'LaTeX with acmart 2020/09/13 v1.73 Typesetting articles for the Association for Computing Machinery and hyperref 2020-05-15 v7.00e Hypertext links for LaTeX', 'producer': 'pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Liv

BadRequestError: Error code: 400 - {'error': {'code': 400, 'message': 'the request exceeds the available context size. try increasing the context size or enable context shift', 'type': 'exceed_context_size_error', 'n_prompt_tokens': 4792, 'n_ctx': 4096}}

In [38]:
from agents.main import agent
from utils.logging import logger

In [39]:
convo_messages = [{"role":"user", "content":"Safaricom PLC"}]

In [40]:
thread = {"configurable": {"thread_id": "1", "recursion_limit": 50}}

In [41]:
for chunk in agent.stream({"messages": convo_messages}, config=thread, stream_mode="values"):
    print(chunk)
    # logger.debug(f"Model response chunk: {chunk}")

{'messages': [HumanMessage(content='Safaricom PLC', additional_kwargs={}, response_metadata={}, id='e690b1e6-4adf-4fa6-b4b2-6bd4fbebb724')], 'supervisor_messages': [], 'raw_notes': [], 'notes': []}
{'messages': [HumanMessage(content='Safaricom PLC', additional_kwargs={}, response_metadata={}, id='e690b1e6-4adf-4fa6-b4b2-6bd4fbebb724'), AIMessage(content='Could you please specify the type of report you are requesting for Safaricom PLC? For example, are you interested in a financial report, a risk assessment, a governance review, or something else?', additional_kwargs={}, response_metadata={}, id='10c90419-f4b3-44e6-902e-bb0011a32ee8')], 'supervisor_messages': [], 'raw_notes': [], 'notes': []}


In [42]:
for chunk in agent.stream({"messages": [{"role":"user", "content":"just the financials please"}]}, config=thread, stream_mode="values"):
    print(chunk)

{'messages': [HumanMessage(content='just the financials please', additional_kwargs={}, response_metadata={}, id='81905b53-e281-4e75-9bad-cedcebe1b1fb')], 'supervisor_messages': [], 'raw_notes': [], 'notes': []}
{'messages': [HumanMessage(content='just the financials please', additional_kwargs={}, response_metadata={}, id='81905b53-e281-4e75-9bad-cedcebe1b1fb'), AIMessage(content="Could you please specify which company you'd like a financial report for? Also, could you tell me which country the company is based in?", additional_kwargs={}, response_metadata={}, id='fbcf4d35-cd42-40e6-89ca-db107eedaad5')], 'supervisor_messages': [], 'raw_notes': [], 'notes': []}
